In [ ]:
from typing import Optional
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
import json
from reviews_api import get_product_rating
import base64 

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

import sqlite3

load_dotenv()

True

In [7]:
DB_path = os.path.join(os.getcwd(), "store.db")

print(DB_path)

d:\ksn-II\PRODUCTION\store.db


In [8]:
llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0) 
vision_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [14]:

@tool
def search_products(query:str, max_price:Optional[float]=None, is_organic:Optional[bool]=None) :
    """
    Search the product database by the keyword (matchd against name , description and category).
    operationally filter by maximum price and/or organic status.
    Return a Json array of matching products, each with :id ,name, category, price,description, is_organic.

    """

    conn = sqlite3.connect(DB_path)
    cursor = conn.cursor()

    sql="SELECT id,name,category,price ,description,is_organic FROM products Where 1=1"
    params : list=[]

    if query:
        sql += " AND name LIKE ?"
        params.append(f"%{query}%")

    if max_price is not None:
        sql += " AND price <= ?"
        params.append(max_price)

    if is_organic is not None:
        sql += " AND is_organic = ?"
        params.append(1 if is_organic else 0)

    cursor.execute(sql,params)
    rows=cursor.fetchall()
    conn.close()


    products=[
        {
            "id": row[0],
            "name": row[1],
            "category":row[2],
            "price": row[3],
            "description":row[4],
            "is_organic": bool(row[5]),
        }
        for row in rows
            

    ] 
    return json.dumps(products)


In [15]:
@tool
def get_rating(product_id:int)->str:
    """
    Get the average customer rating and total review count for a product by its ID. Returns a JSON object
    with: product_id,average_rating,review_count.
    """

    result= get_product_rating(product_id)
    return json.dumps(result)

In [16]:
store={}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id]=InMemoryChatMessageHistory()
    return store[session_id]

In [17]:
@tool
def checkout(product_id: int) -> str:
    """
    Place an order for the given product ID. Saves the order to the database and returns a confirmation message
    with the order ID, product name, and price.
    """
    conn = sqlite3.connect(DB_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name, price FROM products WHERE id = ?", (product_id,))
    row = cursor.fetchone()

    if not row:
        conn.close()
        return f"Error: product with ID {product_id} not found."

    name, price = row
    cursor.execute(
        "INSERT INTO orders (product_id, product_name, price) VALUES (?, ?, ?)",
        (product_id, name, price),
    )
    order_id = cursor.lastrowid
    conn.commit()
    conn.close()
    return f"Order #{order_id} confirmed: {name} — ₹{price}"

In [18]:
@tool 
def describe_product_image(image_path:str)->str:
    """
    Analyze a product image and return its key attributes as a json objects. use this when the user 
    uploads a photo or a product they are interested in. the returned attributes can be used directly 
    with search_products.
    """

    try:
        if image_path.startswith("http://") or image_path.startswith("https://"):
            import requests
            image_data = base64.b64encode(requests.get(image_path).content).decode()
            ext = os.path.splitext(image_path)[1].lower().lstrip(".")
        else:
            with open(image_path,"rb") as f:
                image_data= base64.b64encode(f.read()).decode()
            ext=os.path.splitext(image_path)[1].lower().lstrip(".")
    except (FileNotFoundError, OSError):
        return (
            "Error: no image was found at that path. Ask the user to upload a photo "
            "using the image uploader before trying again."
        )
    mime= "image/jpeg" if ext in ("jpg","jpeg") else f"image/{ext}"

    message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": (
                "Look at this product image and extract its key attributes. "
                "Return ONLY a JSON object with these fields:\n"
                "- product_type: what kind of product it is (e.g. honey, olive oil)\n"
                "- search_query: a short keyword to search for it\n"
                "- is_organic: true if the label says organic, false if not, null if unclear\n"
                "- description: one sentence describing the product"
            ),
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime};base64,{image_data}"
            },
        },
    ]
)

    response = vision_llm.invoke([message])
    return response.content


In [22]:
@tool
def add_to_cart(session_id: str, product_id: int) -> str:
    """
    Use this tool when the user wants to add a product to their shopping cart.
    """

    print("🔥 ADD_TO_CART TOOL CALLED")
    print("Session:", session_id)
    print("Product ID:", product_id)

    conn = sqlite3.connect(DB_path)
    cursor = conn.cursor()



In [ ]:

@tool

def add_to_cart(session_id: str, product_id: int) -> str:

    """
    Adds a product to the user's shopping cart.

    If the product is already in the cart, its quantity is increased by one.
    If the product is not in the cart, it is added with a quantity of one.

    Args:
        session_id: Unique identifier of the user's shopping session.
        product_id: ID of the product to add.

    Returns:
        A message indicating whether the product was added or its quantity was updated.
    """
    conn = sqlite3.connect(DB_path)

    try:
        cursor = conn.cursor()

        cursor.execute(
            "SELECT * FROM products WHERE id = ?",
            (product_id,)
        )
        product = cursor.fetchone()

        if product is None:
            return "❌ Product not found."

        cursor.execute(
            """
            SELECT * FROM cart
            WHERE session_id = ?
            AND product_id = ?
            """,
            (session_id, product_id)
        )

        cart_item = cursor.fetchone()

        if cart_item is None:
            cursor.execute(
                """
                INSERT INTO cart (session_id, product_id)
                VALUES (?, ?)
                """,
                (session_id, product_id)
            )
            message = "✅ Product added to cart."

        else:
            cursor.execute(
                """
                UPDATE cart
                SET quantity = quantity + 1
                WHERE session_id = ?
                AND product_id = ?
                """,
                (session_id, product_id)
            )
            message = "✅ Product quantity updated."

        conn.commit()
        return message

    finally:
        conn.close()

In [23]:
agent = create_agent(
    tools=[search_products, get_rating, checkout,describe_product_image,add_to_cart],
    model=llm,
    system_prompt = """
You are a helpful AI shopping assistant. Your goal is to help users discover products, answer product-related questions, manage their shopping cart, and place orders correctly.

Follow the workflows below exactly.


*IMAGE SEARCH*


When the user provides an image path or uploads an image of a product:

1. Call describe_product_image.
2. Use the returned search_query and is_organic values.
3. Call search_products using those values.
4. Continue with the normal browsing workflow.

Never attempt to identify products yourself if an image is available.


*BROWSING PRODUCTS*


When the user asks to find, search, browse, or recommend products:

Examples:
- Show me honey
- Organic rice
- Best olive oil
- Honey under $20
- Organic snacks

Workflow:

1. Call search_products.
2. Apply every filter mentioned by the user.
   - organic
   - category
   - maximum price
   - minimum price
   - keyword
3. For every returned product, call get_rating.
4. If the user requested a minimum rating, filter the products.
5. Present products using exactly this format:

#1. Organic Raw Honey (ID:1) — $12.99 ★4.8 — Organic

#2. Wild Forest Honey (ID:5) — $14.50 ★4.6 — Non-Organic

Leave one blank line between products.

Always include the product ID exactly as:

(ID:X)

Never remove the ID.

If only one product matches, still display it and ask:

"Would you like to add it to your cart or buy it now?"

Never call checkout during browsing.

Never call add_to_cart during browsing.


*ADD TO CART*


Use add_to_cart whenever the user wants to save a product instead of immediately purchasing it.

Examples:

- Add it to my cart
- Add the first one
- Put number 2 in my cart
- Save this
- I want this later
- Add Organic Honey

Workflow:

1. Determine which product the user is referring to.
2. Find its product ID from the previously displayed product list.
3. Call add_to_cart using:
   - session_id
   - product_id
4. Tell the user the product was added successfully.

Rules:

Never guess the product ID.

If multiple products were shown and the user's choice is unclear, ask for clarification.

If there is no previous product list, ask the user which product they want.


*BUY NOW / CHECKOUT*


Only call checkout when the user clearly wants to purchase.

Examples:

- Buy it
- Order it
- Checkout
- Purchase number 2
- Yes, buy it
- Go ahead
- Place the order

Workflow:

1. Identify the selected product ID from the previous product list.
2. Call checkout with the correct product_id.
3. Confirm the purchase.

Never call checkout unless the user explicitly wants to buy.

Never confuse "Add to cart" with "Buy now".


*GENERAL RULES*


Always use tools whenever possible.

Never invent products.

Never invent ratings.

Never invent prices.

Never invent product IDs.

Always obtain product IDs from search_products.

Always obtain ratings from get_rating.

Always include (ID:X) when displaying products.

If a product cannot be found, politely inform the user.

If the user's request is ambiguous, ask a clarifying question instead of guessing.

Never skip a required tool call.

Keep responses concise, friendly, and helpful.

Never expose internal reasoning or tool details to the user.
"""    
)

In [24]:
agent_with_memory=RunnableWithMessageHistory(
    runnable=agent,
    get_session_history=get_session_history,
    input_messages_key="messages",
)
config = {
    "configurable": {
        "session_id": "john"
    }
}

d:\ksn-II\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
print(add_to_cart.invoke({"session_id": "john", "product_id": 1}))
print(add_to_cart.invoke({"session_id": "john", "product_id": 1}))
print(add_to_cart.invoke({"session_id": "john", "product_id": 2}))

In [39]:
result = search_products.invoke({
    "query": "honey",
    "max_price": None,
    "organic_only": False
})

print(result)

[{"id": 1, "name": "Organic Raw Honey", "category": "honey", "price": 14.99, "description": "Pure organic raw honey, unfiltered and cold-pressed", "is_organic": true}, {"id": 2, "name": "Wildflower Honey", "category": "honey", "price": 12.99, "description": "Natural wildflower honey from local beekeepers", "is_organic": false}, {"id": 3, "name": "Organic Manuka Honey", "category": "honey", "price": 29.99, "description": "Premium organic Manuka honey from New Zealand", "is_organic": true}, {"id": 4, "name": "Clover Honey", "category": "honey", "price": 8.99, "description": "Classic clover honey, smooth and sweet", "is_organic": false}, {"id": 5, "name": "Organic Buckwheat Honey", "category": "honey", "price": 18.99, "description": "Dark and robust organic buckwheat honey, antioxidant-rich", "is_organic": true}, {"id": 6, "name": "Orange Blossom Honey", "category": "honey", "price": 15.99, "description": "Light and floral orange blossom honey", "is_organic": false}, {"id": 7, "name": "Or

In [40]:
print(get_rating.invoke({"product_id": 1}))

{"product_id": 1, "average_rating": 4.62, "review_count": 4}


In [41]:
print(type(describe_product_image))

<class 'langchain_core.tools.structured.StructuredTool'>


In [25]:
response = agent_with_memory.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Show me honey under $20"
            }
        ]
    },
    config=config
)

print(response)

{'messages': [HumanMessage(content='Show me honey under $20', additional_kwargs={}, response_metadata={}, id='8777da5f-38a8-4436-8040-2429949b6a8d'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'p62q0j95f', 'function': {'arguments': '{"max_price":20,"query":"honey"}', 'name': 'search_products'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 1488, 'total_tokens': 1597, 'completion_time': 0.126464984, 'completion_tokens_details': None, 'prompt_time': 0.110898589, 'prompt_tokens_details': None, 'queue_time': 0.050248168, 'total_time': 0.237363573}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7aff-52ac-73a2-99ec-129a29b0023b-0', tool_calls=[{'name': 'search_products', 'args': {'max_price': 20, 'query': 'honey'}, 'id': 'p62q0j95f', 'type': 'tool_call'}], invalid_tool